# 03. Model Baseline
Baseline models implementation.

In [ ]:
import pandas as pd
import os
import notebook_const

from src import utils
from src import const

# Load Split Data using shared function
df_train, df_valid, df_test, df_process, split_info = utils.load_split_data_with_combined()

if df_process is None:
    raise RuntimeError("Data not found. Please run 02_process_data.ipynb first.")

In [ ]:
df_train.head()

In [ ]:
# Create Sequences using shared function (lag-aware)
lag_columns = sorted([col for col in df_process.columns if col.startswith('lag_arrival_delay_')])
requested_past_trips = 5
n_past_trips = min(requested_past_trips, len(lag_columns))
data = utils.prepare_model_data(df_train, df_test, df_process, n_past_trips=n_past_trips)

# Extract variables
X_delays_train, X_features_train, X_agg_train, y_train = \
    data['X_delays_train'], data['X_features_train'], data['X_agg_train'], data['y_train']
X_delays_test, X_features_test, X_agg_test, y_test = \
    data['X_delays_test'], data['X_features_test'], data['X_agg_test'], data['y_test']
n_stops = data['n_stops']

In [ ]:
import os
from datetime import datetime
import numpy as np

split_output_dir = const.SPLITTED_DATA_DIR
os.makedirs(split_output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
lag_label = f'{n_past_trips}lag'

train_file = os.path.join(split_output_dir, f'train_{lag_label}_{timestamp}.npz')
test_file = os.path.join(split_output_dir, f'test_{lag_label}_{timestamp}.npz')
meta_file = os.path.join(split_output_dir, f'meta_{lag_label}_{timestamp}.json')

np.savez_compressed(
    train_file,
    X_delays=X_delays_train,
    X_features=X_features_train,
    X_agg=X_agg_train,
    y=y_train
 )
np.savez_compressed(
    test_file,
    X_delays=X_delays_test,
    X_features=X_features_test,
    X_agg=X_agg_test,
    y=y_test
 )

metadata = {
    'n_past_trips': int(n_past_trips),
    'n_stops': int(data['n_stops']),
    'lag_columns': lag_columns[:n_past_trips],
    'train_file': os.path.basename(train_file),
    'test_file': os.path.basename(test_file)
}
pd.Series(metadata).to_json(meta_file, force_ascii=False, indent=2)

print(f"Saved train tensors to {train_file}")
print(f"Saved test tensors to {test_file}")
print(f"Saved metadata to {meta_file}")

In [ ]:
# Baseline 1: Last Trip Delay
evaluation_results = []

y_pred_baseline1 = X_delays_test[:, -1, :]

result_bl1 = utils.evaluate_model(
    y_test, y_pred_baseline1,
    model_name="Baseline 1 (Last Trip)",
    config={"method": "last_trip", "n_past_trips": n_past_trips}
)
evaluation_results.append(result_bl1)
print(result_bl1.summary())

In [ ]:
# Baseline 2: Mean of Past N Trips
y_pred_baseline2 = X_delays_test.mean(axis=1)

result_bl2 = utils.evaluate_model(
    y_test, y_pred_baseline2,
    model_name="Baseline 2 (Mean Past N)",
    config={"method": "mean_past_n", "n_past_trips": n_past_trips}
)
evaluation_results.append(result_bl2)
print(result_bl2.summary())

In [ ]:
# Model Comparison Table and Save Results
utils.display_and_save_results(evaluation_results, const.EVALUATION_RESULTS_BASELINE)